In [ ]:
!pip install langchain langchain-community langgraph langchain-core langchain-classic langchain[openai] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 2.8 MB/s eta 0:00:00


In [ ]:
import os
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.chat_models import init_chat_model
from langchain_classic import hub
from langgraph.prebuilt import create_react_agent

In [ ]:
db = SQLDatabase.from_uri('sqlite:///data/northwind.db')
print(db.dialect)
print(db.get_usable_table_names())

sqlite
['Categories', 'CustomerCustomerDemo', 'CustomerDemographics', 'Customers', 'EmployeeTerritories', 'Employees', 'Order Details', 'Orders', 'Products', 'Regions', 'Shippers', 'Suppliers', 'Territories']


In [ ]:
# execute sample query
print(db.run("SELECT * from Customers LIMIT 3;"))

[('ALFKI', 'Alfreds Futterkiste', 'Maria Anders', 'Sales Representative', 'Obere Str. 57', 'Berlin', 'Western Europe', '12209', 'Germany', '030-0074321', '030-0076545'), ('ANATR', 'Ana Trujillo Emparedados y helados', 'Ana Trujillo', 'Owner', 'Avda. de la Constitución 2222', 'México D.F.', 'Central America', '05021', 'Mexico', '(5) 555-4729', '(5) 555-3745'), ('ANTON', 'Antonio Moreno Taquería', 'Antonio Moreno', 'Owner', 'Mataderos  2312', 'México D.F.', 'Central America', '05023', 'Mexico', '(5) 555-3932', None)]


In [ ]:
# initialize llm
os.environ["AZURE_OPENAI_API_KEY"] = ""
os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["OPENAI_API_VERSION"] = ""
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = "gpt-4o"

llm = init_chat_model(
    "azure_openai:gpt-4o",
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
)

# initialize the toolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

In [ ]:
print(tools)

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7f1c736c2ea0>), InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7f1c736c2ea0>), ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7f1c736c2ea0>), QuerySQLCheckerTool(description='Use this tool to double check if

In [ ]:
# prompt template for nl2sql
prompt_template = hub.pull('langchain-ai/sql-agent-system-prompt')
prompt_template.messages[0].pretty_print()

================================ System Message ================================

You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to th

In [ ]:
system_message = prompt_template.format(dialect='SQLite', top_k=5)

# create the sql ai agent
sql_agent = create_react_agent(llm, tools, prompt=system_message)

/tmp/ipython-input-992761008.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  sql_agent = create_react_agent(llm, tools, prompt=system_message)


In [ ]:
# sample execution
query = 'Identify the ten most expensive products (name, unit price).'

# stream the events
for event in sql_agent.stream(
    {"messages": ('user', query)},
    stream_mode='values'
):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

Identify the ten most expensive products (name, unit price).
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_FLL7qQwAUiTdrAFSWBYyG7R0)
 Call ID: call_FLL7qQwAUiTdrAFSWBYyG7R0
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Categories, CustomerCustomerDemo, CustomerDemographics, Customers, EmployeeTerritories, Employees, Order Details, Orders, Products, Regions, Shippers, Suppliers, Territories
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_lrN1QTSzPckKwpQxQI8PZ1rh)
 Call ID: call_lrN1QTSzPckKwpQxQI8PZ1rh
  Args:
    table_names: Products
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Products" (
	"ProductID" INTEGER NOT NULL, 
	"Prod

In [ ]:
# sample execution
query = 'Get the top 5 employees and the number of orders they have processed'

# stream the events
for event in sql_agent.stream(
    {"messages": ('user', query)},
    stream_mode='values'
):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

Get the top 5 employees and the number of orders they have processed
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_X1Q5GgeqW2CTohyNd32k5jkv)
 Call ID: call_X1Q5GgeqW2CTohyNd32k5jkv
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Categories, CustomerCustomerDemo, CustomerDemographics, Customers, EmployeeTerritories, Employees, Order Details, Orders, Products, Regions, Shippers, Suppliers, Territories
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_PpUX7wB3xWOrIgzO7tq6IVAI)
 Call ID: call_PpUX7wB3xWOrIgzO7tq6IVAI
  Args:
    table_names: Orders, Employees
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Employees" (
	"EmployeeID" INTEGE